# Human-in-the-Loop [Step 3 - Approval Gates and Human Feedback]

> **MLCourse - Agentic AI - CrewAI Flows and Orchestration**

CrewAI supports human-in-the-loop (HITL) via the `@human_feedback` decorator
on Flow methods and the `human_input` parameter on Tasks. These mechanisms
pause execution and wait for a human to review, approve, or modify the
output before the flow continues.

## What you will learn

1. The `@human_feedback` decorator on Flow methods.
2. The `human_input=True` parameter on Tasks for agent-level HITL.
3. Building approval gates in multi-step flows.
4. Combining flow-level and task-level human input.

## Key takeaways

- `@human_feedback` pauses the flow at that step and shows a prompt.
- The human response is returned as the step's output and flows downstream.
- `human_input=True` on a Task makes the agent ask the human for input.
- HITL is essential for production systems that need human oversight.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os                           # Environment variable access
from pathlib import Path            # OOP path handling

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv      # Load .env into os.environ

TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1 -- Verify Ollama availability

In [ ]:
from langchain_ollama import ChatOllama

try:
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("[GREEN] Ollama reachable -- full pipeline will run")
except Exception as exc:
    LLM_AVAILABLE = False
    print("[WARN] Ollama not reachable:", exc)
    print("Flow structure demonstrated without LLM calls")

## 2 -- Import CrewAI HITL classes

In [ ]:
from pydantic import BaseModel
from crewai import Flow, Agent, Task, Crew
from crewai.flow import start, listen, human_feedback

print("CrewAI HITL imports successful")

## 3 -- Define typed state

Our flow will generate a content brief, ask for human approval, then
proceed to drafting. The state tracks the brief, approval status, and
the final draft.

In [ ]:
class ContentState(BaseModel):
    """State for a content creation flow with human approval."""
    topic: str = ""          # Input topic
    brief: str = ""          # Generated content brief
    approved: bool = False   # Whether the human approved the brief
    feedback: str = ""       # Human feedback on the brief
    draft: str = ""          # Final draft after approval

print("State model defined:", list(ContentState.model_fields.keys()))

## 4 -- Define agents and tasks

In [ ]:
llm = ChatOllama(model="llama3.1:8b", temperature=0) if LLM_AVAILABLE else None

strategist = Agent(
    role="Content Strategist",
    goal="Create detailed content briefs that align with business objectives",
    backstory="You are a senior content strategist with 10 years of experience.",
    llm=llm,
    verbose=False,
)

writer = Agent(
    role="Content Writer",
    goal="Write engaging content based on an approved brief",
    backstory="You are a professional writer who produces polished drafts.",
    llm=llm,
    verbose=False,
)

brief_task = Task(
    description="Create a content brief for the topic: {topic}",
    expected_output="A structured brief with headline, key points, target audience, and tone.",
    agent=strategist,
)

draft_task = Task(
    description="Write a draft based on this approved brief: {brief}",
    expected_output="A polished 200-300 word draft following the brief.",
    agent=writer,
)

print("Agents and tasks defined")

## 5 -- Build a flow with human approval gate

The `@human_feedback` decorator creates a checkpoint where the flow
pauses and waits for human input. The decorator takes a `message` that
is displayed to the human, asking them to approve or provide feedback.

In a notebook environment, `@human_feedback` uses the `InputProvider`
to collect responses. For production, you would configure a web UI or
Slack integration as the provider.

In [ ]:
class ContentCreationFlow(Flow[ContentState]):
    """Flow with a human approval gate between drafting and writing.

    Pipeline: generate brief -> human reviews -> write draft
    """

    @start()
    def generate_brief(self):
        """Step 1: Generate a content brief from the topic."""
        print(f"[Flow] Generating brief for: {self.state.topic}")
        crew = Crew(
            agents=[strategist],
            tasks=[brief_task],
            verbose=False,
        )
        result = crew.kickoff(inputs={"topic": self.state.topic})
        self.state.brief = str(result)
        print(f"[Flow] Brief generated ({len(self.state.brief)} chars)")
        return self.state.brief

    @human_feedback(
        message="Please review the content brief below. Approve it or provide feedback.",
    )
    @listen(generate_brief)
    def review_brief(self, brief_result):
        """Step 2: Human reviews the brief before proceeding.

        The @human_feedback decorator pauses execution here. In a notebook,
        it prompts for input via the configured InputProvider. The human
        response becomes the return value of this method.
        """
        print(f"[Flow] Brief ready for human review ({len(brief_result)} chars)")
        # In a real scenario, this method would display the brief
        # and collect human feedback. The decorator handles the pause.
        return brief_result

    @listen(review_brief)
    def write_draft(self, approved_brief):
        """Step 3: Write the draft based on the approved brief."""
        print(f"[Flow] Writing draft based on approved brief")
        self.state.approved = True
        crew = Crew(
            agents=[writer],
            tasks=[draft_task],
            verbose=False,
        )
        result = crew.kickoff(inputs={"brief": approved_brief})
        self.state.draft = str(result)
        print(f"[Flow] Draft complete ({len(self.state.draft)} chars)")
        return self.state.draft

print("Flow with human approval gate defined")

## 6 -- Run the flow (automated mode)

In a notebook, `@human_feedback` uses the configured `InputProvider`.
For automated testing, you can set a `default_outcome` on the decorator
so the flow continues without waiting for input. This is useful for
development and testing.

In [ ]:
# Redefine with default_outcome for automated testing
class AutoContentCreationFlow(Flow[ContentState]):
    """Flow with human feedback that auto-approves for testing.

    The default_outcome parameter on @human_feedback means the flow
    continues automatically in non-interactive environments.
    """

    @start()
    def generate_brief(self):
        """Step 1: Generate a content brief."""
        print(f"[Flow] Generating brief for: {self.state.topic}")
        if LLM_AVAILABLE:
            crew = Crew(
                agents=[strategist],
                tasks=[brief_task],
                verbose=False,
            )
            result = crew.kickoff(inputs={"topic": self.state.topic})
            self.state.brief = str(result)
        else:
            self.state.brief = f"Auto-generated brief for: {self.state.topic}"
        print(f"[Flow] Brief generated ({len(self.state.brief)} chars)")
        return self.state.brief

    @human_feedback(
        message="Review the brief. Approve or provide feedback.",
        default_outcome="approved",  # Auto-approve in non-interactive mode
    )
    @listen(generate_brief)
    def review_brief(self, brief_result):
        """Step 2: Human review checkpoint."""
        print(f"[Flow] Human review checkpoint (auto-approved)")
        self.state.approved = True
        return brief_result

    @listen(review_brief)
    def write_draft(self, approved_brief):
        """Step 3: Write draft from approved brief."""
        print(f"[Flow] Writing draft")
        if LLM_AVAILABLE:
            crew = Crew(
                agents=[writer],
                tasks=[draft_task],
                verbose=False,
            )
            result = crew.kickoff(inputs={"brief": approved_brief})
            self.state.draft = str(result)
        else:
            self.state.draft = f"Draft based on: {approved_brief[:100]}"
        print(f"[Flow] Draft complete ({len(self.state.draft)} chars)")
        return self.state.draft

print("Auto-approving flow defined for testing")

## 7 -- Execute the auto-approving flow

In [ ]:
flow = AutoContentCreationFlow()
result = flow.kickoff(inputs={"topic": "Why retrieval-augmented generation matters for enterprise AI"})

print("\n" + "=" * 60)
print("FLOW COMPLETE")
print("=" * 60)
print(f"Topic:    {flow.state.topic}")
print(f"Brief:    {flow.state.brief[:200]}...")
print(f"Approved: {flow.state.approved}")
print(f"Draft:    {flow.state.draft[:200]}...")

## 8 -- Task-level human_input

In addition to flow-level `@human_feedback`, individual Tasks support
`human_input=True`. When set, the agent will ask the human for input
during execution. This is useful for:

- Agents that need clarification mid-task.
- Review loops where the agent proposes changes and the human approves.
- Interactive debugging where you want to inspect agent reasoning.

In [ ]:
# Define a task with human_input=True
interactive_task = Task(
    description="Research and summarize: {topic}. Ask the human if you need clarification.",
    expected_output="A summary with any clarifying questions answered.",
    agent=strategist if LLM_AVAILABLE else None,
    human_input=True,  # Agent will ask the human for input during execution
)

print("Task with human_input=True defined")
print("When executed, the agent will pause and ask for human clarification")
print("Note: human_input=True works at the agent/Task level, not the flow level")

## 9 -- Combining flow and task HITL

You can combine both mechanisms in a single flow:

- `@human_feedback` on a Flow method for high-level approval gates.
- `human_input=True` on a Task for agent-level questions.

This gives you two levels of human oversight:
1. **Gate-level**: Human approves or rejects entire steps.
2. **Agent-level**: Human answers questions during agent execution.

In [ ]:
# Demonstrate the combined pattern (structure only -- no LLM calls)
class CombinedHITLFlow(Flow[ContentState]):
    """Flow combining @human_feedback gates with human_input tasks."""

    @start()
    def plan(self):
        """Step 1: Plan the content."""
        print("[Flow] Planning content")
        self.state.brief = f"Plan for: {self.state.topic}"
        return self.state.brief

    @human_feedback(
        message="Does this plan look good? Approve or suggest changes.",
        default_outcome="approved",
    )
    @listen(plan)
    def approve_plan(self, plan_result):
        """Step 2: Human approves the plan."""
        print("[Flow] Plan approved by human")
        self.state.approved = True
        return plan_result

    @listen(approve_plan)
    def execute(self, approved_plan):
        """Step 3: Execute with agent-level human_input."""
        print("[Flow] Executing with human_input=True on task")
        # In production, the Task with human_input=True would
        # pause the agent to ask the human for clarification.
        self.state.draft = f"Execution result for: {approved_plan}"
        return self.state.draft

print("Combined HITL flow defined")

In [ ]:
# Run the combined flow
combined_flow = CombinedHITLFlow()
result = combined_flow.kickoff(inputs={"topic": "AI safety best practices"})

print(f"Plan:     {combined_flow.state.brief}")
print(f"Approved: {combined_flow.state.approved}")
print(f"Result:   {combined_flow.state.draft}")

## 10 -- Summary

CrewAI provides two levels of human-in-the-loop:

| Mechanism | Level | Use Case |
|-----------|-------|----------|
| `@human_feedback` | Flow method | Approval gates, review checkpoints |
| `human_input=True` | Task | Agent questions, mid-task clarification |

Key points:
- `@human_feedback` pauses the flow and collects human input via `InputProvider`.
- `default_outcome` parameter enables auto-approval for testing.
- `human_input=True` on Tasks makes agents ask humans for input.
- Both can be combined for multi-level oversight.

In [ ]:
print("=" * 60)
print("MODULE SUMMARY -- Human-in-the-Loop")
print("=" * 60)
print()
print("Flow-level HITL:")
print("  @human_feedback(message='...', default_outcome='...')")
print("  - Pauses the flow at that step")
print("  - Displays message to human")
print("  - Human response becomes the step's output")
print()
print("Task-level HITL:")
print("  Task(..., human_input=True)")
print("  - Agent asks human for input during execution")
print("  - Useful for clarification and interactive debugging")
print()
print("Combined pattern:")
print("  1. @human_feedback for gate-level approval")
print("  2. human_input=True for agent-level questions")
print("  3. Both work together for multi-level oversight")